# NLP Day 2 · 실습 1. Skip-gram 학습 쌍 만들기

**연계 이론:** 이론 1. 분산표현과 임베딩 · 이론 2. Word2Vec의 학습 원리

## 1. 이번 실습에서 만들 것

Word2Vec의 **Skip-gram**은 중심 단어를 입력으로 사용하여 주변 단어를 예측하면서 단어 임베딩을 학습합니다. 이번 실습에서는 문장에서 **(중심 단어, 주변 단어)** 쌍을 만들고, 작은 신경망에 전달해 임베딩 벡터가 변화하는 과정을 관찰합니다.

| 구분 | 활동 | 확인할 결과 |
|---|---|---|
| A형 | `나는 따뜻한 커피를 마신다`에서 좌우 1칸의 학습 쌍 생성 후 모델 학습 | 학습 쌍 6개, 임베딩·손실 변화 |
| B형 | `나는 따뜻한 차를 마신다`에서 `make_skipgram_pairs()` 완성 | 학습 쌍 6개와 모델 입력 연결 |

**핵심 개념:** 중심 단어와 주변 단어를 학습의 입력·정답으로 만들 수 있습니다.

## 2. A형 - 함께 구현하기

### 2.1 문장을 토큰으로 나누고 학습 쌍 생성하기

`Context Window = 1`은 중심 단어의 **왼쪽 1칸과 오른쪽 1칸**을 확인한다는 뜻입니다. 문장 밖의 위치나 중심 단어 자신은 주변 단어에 포함하지 않습니다.

In [ ]:
import torch
from torch import nn

# 난수 시드를 고정해 실행할 때마다 임베딩 초기값을 재현.
torch.manual_seed(42)

sentence_a = "나는 따뜻한 커피를 마신다"
tokens_a = sentence_a.split()  # 띄어쓰기를 기준으로 단어 4개로 분리.
window_size = 1  # 중심 단어의 왼쪽·오른쪽 1칸을 주변 문맥으로 사용.

pairs_a = []
for center_idx, center_word in enumerate(tokens_a):
    # 문장 밖으로 범위가 벗어나지 않도록 시작·끝 인덱스를 제한.
    # range(start, end)에서는 end 위치가 포함되지 않으므로 +1이 필요.
    start = max(0, center_idx - window_size)
    end = min(len(tokens_a), center_idx + window_size + 1)
    
    for context_idx in range(start, end):
        if context_idx == center_idx:
            continue  # 중심 단어 자신은 주변 단어(정답)가 아니므로 제외.
        # Skip-gram 학습 쌍: (중심 단어=입력, 주변 단어=정답)
        pairs_a.append((center_word, tokens_a[context_idx]))

for i, (center, context) in enumerate(pairs_a, start=1):
    print(f"{i}. 중심: {center:<5} → 주변: {context}")

# 단어 4개에 Window=1이면 양끝에서 1개씩, 가운데에서 2개씩 총 6쌍.
assert len(pairs_a) == 6
print("생성된 학습 쌍:", len(pairs_a), "개")


1. 중심: 나는    → 주변: 따뜻한
2. 중심: 따뜻한   → 주변: 나는
3. 중심: 따뜻한   → 주변: 커피를
4. 중심: 커피를   → 주변: 따뜻한
5. 중심: 커피를   → 주변: 마신다
6. 중심: 마신다   → 주변: 커피를
생성된 학습 쌍: 6 개


### 2.2 학습 쌍을 토큰 ID로 바꾸기

모델의 입력은 중심 단어의 **토큰 ID**, 정답은 주변 단어의 **토큰 ID**입니다. Word2Vec은 문장에 별도로 정답 라벨을 붙이지 않아도 이러한 학습 쌍을 만들 수 있습니다.

In [2]:
# 중복 단어를 제거하면서 처음 등장한 순서를 유지해 단어→정수 ID를 만듦.
vocab_a = {word: idx for idx, word in enumerate(dict.fromkeys(tokens_a))}

# 학습 쌍의 첫 번째 요소(중심 단어)는 모델 입력, 두 번째(주변 단어)는 정답.
input_ids_a = torch.tensor([vocab_a[center] for center, _ in pairs_a], dtype=torch.long)
target_ids_a = torch.tensor([vocab_a[context] for _, context in pairs_a], dtype=torch.long)
# nn.Embedding과 CrossEntropyLoss는 단어 문자열이 아닌 정수 ID(Long Tensor)를 사용.

print("어휘집:", vocab_a)
print("입력 ID:", input_ids_a.tolist())
print("정답 ID:", target_ids_a.tolist())
print("입력 Shape:", tuple(input_ids_a.shape))
print("정답 Shape:", tuple(target_ids_a.shape))

# 각 학습 쌍에 입력 ID와 정답 ID가 하나씩 있으므로 둘 다 길이가 6.
assert input_ids_a.shape == target_ids_a.shape == (6,)


어휘집: {'나는': 0, '따뜻한': 1, '커피를': 2, '마신다': 3}
입력 ID: [0, 1, 1, 2, 2, 3]
정답 ID: [1, 0, 2, 1, 3, 2]
입력 Shape: (6,)
정답 Shape: (6,)


### 2.3 임베딩을 사용하는 작은 Skip-gram 모델

`nn.Embedding`에서 중심 단어의 벡터를 가져오고, `nn.Linear`로 **어휘집의 각 단어에 대한 점수(Logits)**를 계산합니다. `CrossEntropyLoss`는 예측 점수와 주변 단어 ID를 비교합니다.

In [3]:
class TinySkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim=8):
        super().__init__()
        # E: 입력 토큰 ID를 8차원 벡터로 변환하는 학습 가능한 임베딩 행렬.
        #    행렬 크기 = (어휘집 크기, 임베딩 차원)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # W: 중심 단어의 벡터를 어휘집의 각 단어에 대한 점수(Logits)로 변환.
        #    bias=False는 이 예제를 단순한 기본 Skip-gram 구조로 유지하기 위한 설정.
        self.output_layer = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, center_ids):
        # 중심 단어 ID (학습 쌍 수,) → 임베딩 벡터 (학습 쌍 수, 임베딩 차원)
        vectors = self.embedding(center_ids)
        # 임베딩 벡터 → 각 주변 단어 후보의 예측 점수 (학습 쌍 수, 어휘집 크기)
        logits = self.output_layer(vectors)
        return logits  # 확률이 아닌 점수이며, 손실 함수가 내부적으로 Softmax를 처리.


def train_skipgram(pairs, vocab, epochs=100):
    """제공 코드: 학습 쌍으로 작은 Skip-gram 모델을 학습합니다."""
    torch.manual_seed(42)
    model = TinySkipgram(len(vocab), embedding_dim=8)
    loss_fn = nn.CrossEntropyLoss()  # 예측 Logits와 정답 주변 단어 ID를 비교.
    optimizer = torch.optim.SGD(model.parameters(), lr=0.15)  # E와 W를 함께 갱신.

    # 입력은 중심 단어 ID, 정답은 해당 쌍의 주변 단어 ID로 준비.
    centers = torch.tensor([vocab[c] for c, _ in pairs], dtype=torch.long)
    contexts = torch.tensor([vocab[w] for _, w in pairs], dtype=torch.long)

    # 학습 전 상태를 별도 복사해 두어 학습 후 임베딩과 비교.
    # no_grad(): 변화 관찰용 값은 역전파 계산 그래프를 만들 필요가 없습니다.
    with torch.no_grad():
        before = model.embedding.weight.detach().clone()
        initial_logits = model(centers)

    losses = []
    for _ in range(epochs):
        logits = model(centers)  # 순전파: 중심 단어별 주변 단어 예측 점수 계산
        loss = loss_fn(logits, contexts)  # 정답 주변 단어 ID와 비교해 손실 계산
        optimizer.zero_grad()  # 이전 반복에서 남은 기울기를 초기화
        loss.backward()  # 역전파: E와 W에 대한 기울기 계산
        optimizer.step()  # 가중치 갱신: 다음 예측에서 손실을 줄이는 방향으로 학습
        losses.append(float(loss.item()))  # 학습 전후의 손실 변화를 확인하기 위해 저장

    with torch.no_grad():
        after = model.embedding.weight.detach().clone()
    return model, before, after, losses, initial_logits

model_a, before_a, after_a, losses_a, logits_a = train_skipgram(pairs_a, vocab_a)
print("학습 전 어휘별 점수 Shape:", tuple(logits_a.shape))
print("임베딩 행렬 Shape:", tuple(after_a.shape))
print(f"손실 변화: {losses_a[0]:.4f} → {losses_a[-1]:.4f}")
print("'따뜻한' 학습 전 벡터:", before_a[vocab_a['따뜻한']].tolist())
print("'따뜻한' 학습 후 벡터:", after_a[vocab_a['따뜻한']].tolist())

# 학습 쌍 6개 × 후보 단어 4개 → (6, 4), 어휘집 4개 × 임베딩 차원 8 → (4, 8)
assert logits_a.shape == (6, 4)
assert after_a.shape == (4, 8)
assert not torch.allclose(before_a, after_a)  # 학습으로 임베딩 가중치가 실제로 바뀌었는지 확인


학습 전 어휘별 점수 Shape: (6, 4)
임베딩 행렬 Shape: (4, 8)
손실 변화: 1.8716 → 0.4874
'따뜻한' 학습 전 벡터: [-0.7521352767944336, 1.6487230062484741, -0.3924786448478699, -1.4036071300506592, -0.7278813123703003, -0.5594301819801331, -0.7688388824462891, 0.7624453902244568]
'따뜻한' 학습 후 벡터: [-0.9433724284172058, 1.799567461013794, -0.5363281965255737, -1.3989336490631104, -0.7870851159095764, -0.47998571395874023, -0.9562028646469116, 0.7221581935882568]


**A형 결과 해석**

- 학습 쌍 6개를 한 번에 넣으면 어휘별 점수의 Shape는 `(6, 4)`입니다. 첫 번째 축은 학습 쌍 6개, 두 번째 축은 가능한 정답 단어 4개입니다.
- 임베딩 행렬의 Shape는 `(4, 8)`입니다. 어휘집에 4개 단어가 있고 각 단어를 8차원 벡터로 표현하기 때문입니다.
- 학습 전후의 벡터값과 손실을 비교해 보세요. 작은 문장 한 개의 결과를 일반적인 의미 유사도 성능으로 해석할 필요는 없습니다.

## 3. B형 — 직접 완성하기

다음 문장으로 **`make_skipgram_pairs()`** 함수를 완성합니다.

> **문제:** `나는 따뜻한 차를 마신다`를 `Context Window = 1`로 처리합니다. 각 중심 위치에서 왼쪽·오른쪽 이웃 단어를 찾아 `(중심, 주변)` 쌍을 추가하세요.

주어진 반복문 뼈대를 활용합니다. **중심 단어 자신을 건너뛰는 조건**과 **학습 쌍을 추가하는 코드**만 직접 작성하면 됩니다.

In [4]:
def make_skipgram_pairs(tokens, window_size=1):
    pairs = []
    for center_idx, center in enumerate(tokens):
        # 중심 단어마다 Context Window 범위를 구합니다(문장 경계 밖은 제외).
        start = max(0, center_idx - window_size)
        end = min(len(tokens), center_idx + window_size + 1)
        for context_idx in range(start, end):
            # ★ TODO 1. 중심 단어 자신의 위치라면 건너뛰세요.
            # ★ TODO 2. (중심 단어, 주변 단어) 쌍을 pairs에 추가하세요.
            pass
    return pairs

sentence_b = "나는 따뜻한 차를 마신다"
tokens_b = sentence_b.split()
pairs_b = make_skipgram_pairs(tokens_b, window_size=1)

print("학생이 생성한 학습 쌍:", pairs_b)
print("생성된 학습 쌍 수:", len(pairs_b))

학생이 생성한 학습 쌍: []
생성된 학습 쌍 수: 0


### 3.1 B형 결과 확인 및 모델에 전달하기

함수를 완성했다면 학습 쌍은 6개가 나와야 합니다. 제공된 학습 함수를 다시 사용해 학습 쌍이 모델에 입력되는 과정을 확인합니다.

In [5]:
expected_b = [
    ("나는", "따뜻한"),
    ("따뜻한", "나는"),
    ("따뜻한", "차를"),
    ("차를", "따뜻한"),
    ("차를", "마신다"),
    ("마신다", "차를"),
]

if pairs_b != expected_b:
    print("B형 코드를 완성하면 위 문장에서 학습 쌍 6개가 순서대로 생성됩니다.")
    print("중심 위치를 건너뛰고, 왼쪽·오른쪽 이웃을 추가했는지 확인하세요.")
else:
    vocab_b = {word: idx for idx, word in enumerate(dict.fromkeys(tokens_b))}
    model_b, before_b, after_b, losses_b, logits_b = train_skipgram(pairs_b, vocab_b)
    print("학습 쌍 확인: 통과")
    print("모델 입력 쌍 수:", len(pairs_b))
    print("모델 출력 Shape:", tuple(logits_b.shape))
    print(f"B형 손실 변화: {losses_b[0]:.4f} → {losses_b[-1]:.4f}")
    print("'차를' 임베딩이 변경되었나요?", not torch.allclose(before_b, after_b))
    assert logits_b.shape == (6, 4)

B형 코드를 완성하면 위 문장에서 학습 쌍 6개가 순서대로 생성됩니다.
중심 위치를 건너뛰고, 왼쪽·오른쪽 이웃을 추가했는지 확인하세요.


## 4. 결과 확인과 이번 내용 간단하게 요약

**완료 기준:** B형에서 학습 쌍 6개를 생성하고, '따뜻한'의 주변 단어가 '나는'과 '차를'임을 확인합니다. 중심 단어 자신과 문장 바깥 위치는 포함하지 않습니다. 학습 쌍을 모델에 넣었을 때 `(6, 4)` 크기의 어휘별 점수가 출력되어야 합니다.

결과를 보고 다음을 짧게 설명해 보세요.

- 문장에 별도의 라벨이 없는데도 학습 정답을 만들 수 있는 이유는 무엇인가요?
- 임베딩의 값은 학습 전후에 왜 달라지나요?

**이번 내용 간단하게 요약**

첫째, Skip-gram은 중심 단어를 입력으로 사용하고 주변 단어를 정답으로 사용합니다.

둘째, Context Window의 크기에 따라 생성되는 학습 쌍이 달라지며, 문장 밖의 위치와 중심 단어 자신은 제외합니다.

셋째, 예측 손실을 줄이도록 학습하면서 단어의 임베딩 가중치가 갱신됩니다. 다음 실습에서는 이렇게 학습된 **단어 벡터의 유사도**를 비교합니다.

## 5. B-2 확장 과제 — Skip-gram 학습 데이터 생성 함수 직접 구현하기

### 목표: 문장과 window_size를 받아 학습 쌍을 만들고, 어휘집과 입력/정답 ID 텐서까지 준비하는 build_skipgram_dataset() 함수를 직접 작성합니다.

완성해야 할 내용
1. 문장의 각 단어를 중심 단어로 선택하는 반복문
2. Context Window의 시작·끝 위치 계산(문장 경계 고려)
3. 중심 단어 자신의 위치를 제외하고 (중심 단어, 주변 단어) 학습 쌍 저장
4. 중복 단어를 제거하고 등장 순서를 유지한 어휘집 생성
5. 중심 단어 ID와 주변 단어 ID를 각각 torch.long 텐서로 변환
결과 확인: 나는 따뜻한 차를 마신다에서 Window Size 1이면 6쌍, 2이면 10쌍이 만들어져야 합니다. Window Size 2에서 모델 출력 Shape는 (10, 4)입니다.

힌트: A형의 셀 2와 셀 4를 참고하되, 이번에는 하나의 함수로 결합하세요. 새로운 신경망 모델을 만들 필요는 없습니다.

In [ ]:
# B-2 확장: 다음 함수 내부 TODO 1~5를 직접 완성하세요.
def build_skipgram_dataset(sentence, window_size):
    tokens = sentence.split()
    pairs = []

    # TODO 1. enumerate()를 사용해 각 토큰을 중심 단어로 순회합니다.
    # TODO 2. start/end를 계산합니다. 문장의 경계를 벗어나지 않게 하세요.
    # TODO 3. 주변 위치를 순회하고 중심 위치 자신을 제외한 학습 쌍을 추가합니다.
    # TODO 4. 단어를 등장 순서대로 0부터 ID로 매핑하는 vocab을 만듭니다.
    # TODO 5. pairs를 이용해 input_ids와 target_ids를 torch.long 텐서로 만듭니다.
    raise NotImplementedError("TODO 1~5를 완성한 뒤 이 줄을 삭제하세요.")

    return pairs, vocab, input_ids, target_ids


# 아래의 확인 코드는 수정하지 않습니다.
sentence_b2 = "나는 따뜻한 차를 마신다"
pairs_w1, vocab_w1, input_w1, target_w1 = build_skipgram_dataset(sentence_b2, window_size=1)
pairs_w2, vocab_w2, input_w2, target_w2 = build_skipgram_dataset(sentence_b2, window_size=2)

print("Window=1 학습 쌍 수:", len(pairs_w1))
print("Window=2 학습 쌍 수:", len(pairs_w2))
print("Window=2 입력 ID:", input_w2.tolist())
print("Window=2 정답 ID:", target_w2.tolist())
print("Window=2 입력/정답 Shape:", tuple(input_w2.shape), tuple(target_w2.shape))

assert len(pairs_w1) == 6
assert len(pairs_w2) == 10
assert vocab_w1 == vocab_w2 == {"나는": 0, "따뜻한": 1, "차를": 2, "마신다": 3}
assert input_w2.shape == target_w2.shape == (10,)
assert all(vocab_w2[c] == x and vocab_w2[w] == y
           for (c, w), x, y in zip(pairs_w2, input_w2.tolist(), target_w2.tolist()))

# 기존 A형에서 정의한 학습 함수에 새로 만든 학습 데이터를 전달합니다.
model_w2, before_w2, after_w2, losses_w2, logits_w2 = train_skipgram(pairs_w2, vocab_w2)
print("Window=2 모델 출력 Shape:", tuple(logits_w2.shape))
print(f"Window=2 손실 변화: {losses_w2[0]:.4f} → {losses_w2[-1]:.4f}")
assert logits_w2.shape == (10, 4)
print("B-2 완료: 모든 검증을 통과했습니다.")
